# 📖 Lab 2: Sliding Window Counter

In Lab 1 we learned about the **Token Bucket** algorithm — a way to control how many requests a client can make. Now we'll look at a completely different family of rate limiting algorithms: **window-based counters**.

The idea is simple: divide time into windows (like 1-minute buckets) and count requests in each window. If the count exceeds the limit, reject the request.

We'll start with the simplest version — the **Fixed Window Counter** — see its flaw, and then fix it with the **Sliding Window Counter**.

## Learning Objectives

By the end of this notebook, you'll understand:

- 🪟 How the **Fixed Window Counter** works and its boundary problem
- 🔄 How the **Sliding Window Counter** fixes this with a weighted average
- 📊 How to compare accuracy vs memory vs complexity of different algorithms
- 🤔 When to choose which rate limiting algorithm

## 🛠️ Setup

### 1. Start the services

Open a terminal in the `06-system-designs/rate-limiter/` folder and run:

```bash
docker compose up -d
```

This starts **Redis** (port 6381) and **RedisInsight** (port 5541) for visualization.

### 2. Select the notebook kernel

1. Make sure you have a virtual environment set up:  
   ```bash
   cd 06-system-designs/rate-limiter
   uv venv
   source .venv/bin/activate
   uv sync
   ```
2. In VS Code, click the **kernel picker** (top-right of this notebook).
3. Select the `.venv` kernel from the `rate-limiter` folder.
4. If the kernel doesn't appear, reload the VS Code window (`Cmd+Shift+P` → "Reload Window").

## 🪟 Fixed Window Counter — The Simple Approach

The Fixed Window Counter is the simplest rate limiting algorithm you can imagine:

1. **Divide time into fixed windows** — for example, every minute is one window
2. **Count requests** in the current window
3. **If the count exceeds the limit** → reject the request
4. **When a new window starts** → reset the counter to zero

Here's what it looks like on a timeline:

```
Time:   |--- Window 1 ---|--- Window 2 ---|--- Window 3 ---|
Limit:       100              100              100
Count:        87               42              100 → reject!
```

Simple! Each window is independent. When a new window starts, everyone gets a fresh counter.

But there's a sneaky problem hiding at the **boundaries** between windows...

In [ ]:
import time

class FixedWindowCounter:
    """Count requests in fixed time windows.
    
    Problem: A user can make 100 requests at 12:00:59 and another
    100 at 12:01:00 — getting 200 requests in 2 seconds!
    """
    
    def __init__(self, max_requests: int, window_seconds: int):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self.counters = {}  # {window_key: count}
    
    def _get_window_key(self) -> int:
        """Get the current time window identifier.
        
        Integer division groups all timestamps in the same window
        into the same key. For example, with 60-second windows:
          - time 120.5 → key 2
          - time 135.9 → key 2  (same window!)
          - time 180.0 → key 3  (new window)
        """
        return int(time.time()) // self.window_seconds
    
    def allow_request(self, client_id: str) -> bool:
        """Check if a request from this client is allowed."""
        window = self._get_window_key()
        key = f"{client_id}:{window}"
        
        # Clean up old windows to save memory
        current_keys = [k for k in self.counters if k.startswith(f"{client_id}:")]
        for k in current_keys:
            if k != key:
                del self.counters[k]
        
        count = self.counters.get(key, 0)
        if count >= self.max_requests:
            return False
        
        self.counters[key] = count + 1
        return True

# Create: 5 requests per 10-second window
limiter = FixedWindowCounter(max_requests=5, window_seconds=10)

print("Fixed Window Counter: 5 requests per 10-second window")
print()
for i in range(7):
    allowed = limiter.allow_request("alice")
    status = "✅ ALLOWED" if allowed else "❌ DENIED"
    print(f"  Request {i+1}: {status}")

## ⚠️ The Boundary Problem

Fixed Window Counter has a critical flaw: **traffic spikes at window boundaries**.

Here's a concrete example:

- **Limit**: 100 requests per minute
- A user sends **100 requests** at `12:00:58–12:00:59` → all allowed (end of Window 1)
- The same user sends **100 requests** at `12:01:00–12:01:02` → all allowed (start of Window 2!)
- **Result**: 200 requests in just 4 seconds — **2× the intended limit!**

```
Window 1                    Window 2
|........................===|===........................|
                        ↑   ↑
                   100 here  100 here
                   = 200 in 4 seconds! 💥
```

The user exploited the **boundary** between two windows. Each window sees its own counter (both under the limit), but in reality the traffic burst is way above what we intended.

Let's see this in action:

In [ ]:
import time

def sleep_until_just_before_boundary(window: float, margin: float = 0.15) -> None:
    """Block until `margin` seconds before the next window boundary.

    Windows are keyed off `time.time() // window`, so `time.time() % window` is
    how far into the current window we are, and `window - that` is how long
    until it flips. If we are already inside the margin we wait for the window
    after (a negative sleep is a ValueError, and landing mid-burst on the
    boundary would scramble the demo).
    """
    wait = (window - (time.time() % window)) - margin
    if wait < 0:
        wait += window
    time.sleep(wait)


WINDOW = 2
LIMIT = 5
limiter = FixedWindowCounter(max_requests=LIMIT, window_seconds=WINDOW)

print("=== Demonstrating the Boundary Problem ===")
print(f"Limit: {LIMIT} requests per {WINDOW}-second window\n")

sleep_until_just_before_boundary(WINDOW)

burst_start = time.time()
allowed_before = sum(limiter.allow_request("alice") for _ in range(LIMIT))
print(f"  End of window 1:   {allowed_before}/{LIMIT} allowed")

time.sleep(0.3)   # cross the boundary — every counter resets to zero

allowed_after = sum(limiter.allow_request("alice") for _ in range(LIMIT))
burst_span = time.time() - burst_start
print(f"  Start of window 2: {allowed_after}/{LIMIT} allowed")

total = allowed_before + allowed_after
print(f"\n  Measured: {total} requests allowed in {burst_span:.2f}s")
print(f"  Intended: at most {LIMIT} in any {WINDOW}s stretch")
print(f"  Overshoot: {total / LIMIT:.1f}x the limit")

# This is the *whole reason* sliding windows exist, so make it a hard check
# rather than a claim in a print statement.
assert total == 2 * LIMIT, f"expected a 2x boundary burst, measured {total}"
print("\n  ⚠️  Exactly 2x. Both windows were individually 'under the limit'.")

## 🔄 Sliding Window Counter — The Fix

The **Sliding Window Counter** solves the boundary problem with a clever trick:

Instead of looking at one fixed window, we keep counters for **both** the current window and the previous window. Then we calculate a **weighted average** based on how far we are into the current window.

### The Formula

```
weighted_count = (previous_window_count × overlap%) + current_window_count
```

- **overlap%** = how much of the previous window still "overlaps" with our sliding window
- The further we are into the current window, the less the previous window matters

### Example

Imagine we're **30% into** the current minute:

```
         Previous Window              Current Window
    |=========================|==========....................|
                              ↑          ↑
                         window start   we are here (30%)
    
    |---- 70% overlap -------|--- 30% ---|
    ← our sliding window covers this →
```

- Previous minute had **80 requests**
- Current minute has **20 requests** so far
- overlap% = 1.0 - 0.30 = **0.70** (70% of previous window still overlaps)
- Weighted count = (80 × 0.70) + 20 = 56 + 20 = **76**
- If limit is 100 → still **24 requests remaining** ✅

This is an **approximation** — it assumes traffic was evenly distributed within each window. In practice, this is good enough for most systems.

In [ ]:
import time

class SlidingWindowCounter:
    """A sliding window counter that avoids the boundary problem.
    
    Uses a weighted average of the current and previous window counts
    to approximate a true sliding window. Much more accurate than
    fixed windows, with only slightly more memory (2 counters per client).
    """
    
    def __init__(self, max_requests: int, window_seconds: int):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        # Store: {client_id: {"prev_count": N, "curr_count": N, "curr_window": W}}
        self.state = {}
    
    def _get_window(self) -> int:
        """Get the current window number."""
        return int(time.time()) // self.window_seconds
    
    def allow_request(self, client_id: str) -> bool:
        """Check if a request is allowed using sliding window estimation."""
        current_window = self._get_window()
        now = time.time()
        
        # How far are we into the current window? (0.0 to 1.0)
        window_progress = (now % self.window_seconds) / self.window_seconds
        
        if client_id not in self.state:
            self.state[client_id] = {
                "prev_count": 0,
                "curr_count": 0,
                "curr_window": current_window,
            }
        
        s = self.state[client_id]
        
        # If we've moved to a new window, rotate counters
        if current_window != s["curr_window"]:
            if current_window == s["curr_window"] + 1:
                # Moved exactly one window forward
                s["prev_count"] = s["curr_count"]
            else:
                # Skipped windows (client was idle)
                s["prev_count"] = 0
            s["curr_count"] = 0
            s["curr_window"] = current_window
        
        # Calculate weighted count
        # The further we are into the current window,
        # the less the previous window matters
        overlap = 1.0 - window_progress
        weighted_count = (s["prev_count"] * overlap) + s["curr_count"]
        
        if weighted_count >= self.max_requests:
            return False
        
        s["curr_count"] += 1
        return True
    
    def get_status(self, client_id: str) -> dict:
        """Get current state for debugging."""
        s = self.state.get(client_id, {"prev_count": 0, "curr_count": 0})
        now = time.time()
        window_progress = (now % self.window_seconds) / self.window_seconds
        overlap = 1.0 - window_progress
        weighted = (s["prev_count"] * overlap) + s.get("curr_count", 0)
        return {
            "prev_count": s["prev_count"],
            "curr_count": s.get("curr_count", 0),
            "window_progress": f"{window_progress:.0%}",
            "weighted_count": round(weighted, 1),
        }

# Create: 5 requests per 5-second window
limiter = SlidingWindowCounter(max_requests=5, window_seconds=5)

print("Sliding Window Counter: 5 requests per 5-second window")
print()
for i in range(7):
    allowed = limiter.allow_request("alice")
    status = "✅ ALLOWED" if allowed else "❌ DENIED"
    state = limiter.get_status("alice")
    print(f"  Request {i+1}: {status}  (weighted count: {state['weighted_count']})")

## 📊 Comparing the Two Approaches

Let's put the Fixed Window and Sliding Window Counter side by side:

| Feature | Fixed Window | Sliding Window Counter |
|---------|-------------|------------------------|
| **Memory** | 1 counter per client | 2 counters per client |
| **Accuracy** | ❌ Boundary spike (up to 2× limit) | ✅ Much better (weighted approximation) |
| **Complexity** | Very simple | Slightly more complex |
| **Best for** | Simple, non-critical APIs | Most production systems |

The Sliding Window Counter uses barely any extra memory (just one more counter per client) but dramatically improves accuracy. That's a great trade-off!

Let's prove it with a side-by-side test:

In [ ]:
import time

WINDOW = 2
LIMIT = 5

print("=== Side-by-side: Fixed Window vs Sliding Window ===")
print(f"Both set to: {LIMIT} requests per {WINDOW}-second window\n")

fixed = FixedWindowCounter(max_requests=LIMIT, window_seconds=WINDOW)
sliding = SlidingWindowCounter(max_requests=LIMIT, window_seconds=WINDOW)

sleep_until_just_before_boundary(WINDOW)

# Collect first, print afterwards — printing inside the loop is slow enough to
# eat into the margin before the boundary and make the demo non-deterministic.
phase1 = [(fixed.allow_request("alice"), sliding.allow_request("alice"))
          for _ in range(LIMIT)]

time.sleep(0.3)   # cross the boundary

phase2 = [(fixed.allow_request("alice"), sliding.allow_request("alice"))
          for _ in range(LIMIT)]

for name, phase in [("Phase 1: near the END of a window", phase1),
                    ("Phase 2: at the START of the next window", phase2)]:
    print(name)
    for i, (f, s) in enumerate(phase, 1):
        print(f"  Request {i}:  Fixed={'✅' if f else '❌'}  Sliding={'✅' if s else '❌'}")
    print()

fixed_total = sum(f for f, _ in phase1 + phase2)
sliding_total = sum(s for _, s in phase1 + phase2)

print(f"📊 Fixed Window   allowed {fixed_total}/{2*LIMIT}  "
      f"({fixed_total/LIMIT:.1f}x the limit)")
print(f"📊 Sliding Window allowed {sliding_total}/{2*LIMIT}  "
      f"({sliding_total/LIMIT:.1f}x the limit)")

assert fixed_total > sliding_total, "the boundary burst did not reproduce"
print(f"""
  The sliding window still let {sliding_total - LIMIT} extra through — it is an
  approximation, not a hard guarantee. It assumes the previous window's traffic
  was spread evenly, and here it was all crammed into the final moments, so the
  weighted estimate under-counts. That residual error is the price of storing
  2 numbers per client instead of every timestamp.""")

## 🎯 Sliding Window Log — Perfect Accuracy (at a Cost)

The Sliding Window **Counter** we just built is an *approximation* — it assumes requests are spread evenly within a window. That is usually fine, but sometimes you want **exact accuracy**.

The **Sliding Window Log** keeps a list of the **timestamp of every request**. To decide if a new request is allowed, it simply counts how many timestamps fall inside the last `N` seconds. No approximation — the answer is always exact.

```
now = 12:00:10,   window = 10 seconds
Timestamps stored: [12:00:02, 12:00:05, 12:00:07, 12:00:09]
                    (all within the last 10 seconds)
                    count = 4  ->  allow if limit > 4
```

### Trade-off

| | Sliding Window Counter | Sliding Window Log |
|---|------------------------|---------------------|
| Memory per client | **2 numbers** | **1 number per request** (can be millions!) |
| Accuracy | ~1% error | Perfect |
| Good for | Most APIs | Strict fairness (billing, expensive ops) |

In Redis this log is usually stored as a **sorted set** where score = timestamp.


In [ ]:
import time
from collections import deque


class SlidingWindowLog:
    """Sliding window with a log of every request timestamp.

    Perfect accuracy, but memory grows with the request rate.
    """

    def __init__(self, max_requests: int, window_seconds: float):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self.logs = {}  # {client_id: deque of timestamps}

    def allow_request(self, client_id: str) -> bool:
        now = time.time()
        log = self.logs.setdefault(client_id, deque())

        # Evict timestamps older than the window
        cutoff = now - self.window_seconds
        while log and log[0] <= cutoff:
            log.popleft()

        if len(log) >= self.max_requests:
            return False

        log.append(now)
        return True

    def current_count(self, client_id: str) -> int:
        """Return the current log size (after evicting stale entries)."""
        log = self.logs.get(client_id)
        if not log:
            return 0
        cutoff = time.time() - self.window_seconds
        while log and log[0] <= cutoff:
            log.popleft()
        return len(log)


# Demo
limiter = SlidingWindowLog(max_requests=5, window_seconds=2)

print("Sliding Window Log: 5 requests per 2-second window")
print()
print("Phase 1: Send 5 quick requests (log fills up)")
for i in range(5):
    allowed = limiter.allow_request("alice")
    mark = "ALLOWED" if allowed else "DENIED "
    print(f"  Request {i+1}: {mark}  (log size: {limiter.current_count('alice')})")

print()
print("  The 6th request is denied immediately:")
mark = "ALLOWED" if limiter.allow_request("alice") else "DENIED "
print(f"  Request 6: {mark}")

print()
print("Phase 2: Wait 2.1s -- all old timestamps expire")
time.sleep(2.1)
print(f"  log size after wait: {limiter.current_count('alice')} (old entries were evicted)")

print()
print("Phase 3: Send 5 more requests -- all allowed again")
for i in range(5):
    allowed = limiter.allow_request("alice")
    mark = "ALLOWED" if allowed else "DENIED "
    print(f"  Request {i+1}: {mark}  (log size: {limiter.current_count('alice')})")

print()
print("Notice: the count is ALWAYS exact. No boundary approximation needed.")
print("Downside: each request stores a timestamp -- 1M req/min = 1M timestamps in memory.")


## 🗺️ Algorithm Comparison — The Full Picture

Now that we've learned two algorithms (Token Bucket from Lab 1 and window-based counters from this lab), let's see how all the major rate limiting algorithms compare:

| Algorithm | Memory | Accuracy | Burst Handling | Complexity |
|-----------|--------|----------|----------------|------------|
| **Fixed Window** | 1 counter | ❌ Boundary spike | ❌ None | Very simple |
| **Sliding Window Log** | 1 timestamp/req | ✅ Perfect | ✅ Precise | High memory |
| **Sliding Window Counter** | 2 counters | 🟡 Approximation | 🟡 Good | Moderate |
| **Token Bucket** | 2 values | ✅ Precise | ✅ Natural burst support | Simple |

### When to use which?

- 🪣 **Token Bucket**: Best all-rounder — used by **Stripe**, **AWS**, and many others. Great for APIs that want to allow short bursts of traffic.
- 🔄 **Sliding Window Counter**: Great when you need **window-based counting** (e.g., "100 requests per minute"). Used by **Cloudflare** and others.
- 🪟 **Fixed Window**: Only for non-critical, very simple use cases where the boundary problem is acceptable.
- 📜 **Sliding Window Log**: When you need **perfect accuracy** and memory isn't a concern (stores every single request timestamp).

In the real world, most production systems use either **Token Bucket** or **Sliding Window Counter**.

## 💡 Key Takeaways

1. **Fixed Window Counter** is simple but has a boundary problem — a user can get up to **2× the intended limit** by sending requests at the edge of two windows.

2. **Sliding Window Counter** fixes this with a **weighted average** of the current and previous window counts.

3. The approximation assumes **even traffic distribution** within each window — this is good enough for the vast majority of real-world systems.

4. Memory is minimal: just **2 counters per client** (compared to 1 for Fixed Window).

5. In production, most systems use either **Token Bucket** (Lab 1) or **Sliding Window Counter** (this lab) — they offer the best trade-offs between accuracy, memory, and complexity.

## 🚀 What's Next?

So far, both our labs have run rate limiters **in-memory** — inside a single Python process. But what happens when you have **multiple servers** handling requests?

In the next notebook, we'll make our rate limiter **distributed** using **Redis**. We'll see how to:
- Store rate limit state in Redis so all servers share it
- Use Lua scripts for atomic operations
- Handle the challenges of distributed rate limiting

See you there! 🎉